# ACAC Colab Smoke Test

This notebook checks the RL-facing scheduler interface, async rollout collector, and PyTorch actor/critic forward pass in Colab. It is intentionally small: no training yet, just a clean bridge between `SchedulerEnv` and the ACAC components.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/exGDGD/RL_team.git"
REPO_DIR = Path("/content/RL_team")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(Path.cwd())

In [ ]:
# Colab usually ships with torch already installed, so the repo's local
# requirements intentionally keep torch out. This installs the simulator stack.
!python -m pip install -q -r requirements.txt

import torch
print("torch", torch.__version__)

In [ ]:
# Run only the RL-facing tests first. These include network tests when torch is present.
!python -m pytest tests/test_rl_obs.py tests/test_rl_buffer.py tests/test_rl_rollout.py tests/test_rl_networks.py tests/test_rl_trainer.py -q

In [ ]:
from src.env import CoreType, SchedulerEnv, WorkloadScenario
from src.rl import AgentBatch, build_agent_batch, collect_episode

env = SchedulerEnv(
    core_config={CoreType.P: 1, CoreType.E: 1},
    workload_scenario=WorkloadScenario.BALANCED,
    arrival_rate=0.5,
    episode_time=30.0,
    max_tasks=4,
    seed=3,
)
observations, info = env.reset()
batch = build_agent_batch(observations, agent_order=env.agents)

print("agents:", batch.agent_ids)
print("self:", batch.self_features.shape)
print("ready_queue:", batch.ready_queue.shape)
print("other_cores:", batch.other_cores.shape)
print("action_mask:", batch.action_mask.shape)
print("decision agents:", batch.decision_agent_ids())

In [ ]:
class FirstValidPolicy:
    def act(self, batch: AgentBatch):
        actions = {}
        log_probs = {}
        for row, agent_id in enumerate(batch.agent_ids):
            valid = [
                idx
                for idx, is_valid in enumerate(batch.action_mask[row])
                if idx > 0 and is_valid
            ]
            actions[agent_id] = valid[0] if bool(batch.decision_mask[row]) and valid else 0
            log_probs[agent_id] = 0.0
        return actions, log_probs

env = SchedulerEnv(
    core_config={CoreType.P: 1, CoreType.E: 1},
    workload_scenario=WorkloadScenario.BALANCED,
    arrival_rate=0.5,
    episode_time=30.0,
    max_tasks=4,
    seed=3,
)
buffer = collect_episode(env, FirstValidPolicy(), seed=3)

print("transitions:", len(buffer))
print("completed tasks:", len(env.completed_tasks))
print("first transition elapsed_time:", buffer.transitions[0].elapsed_time)

In [ ]:
import torch

from src.rl.networks import AgentCentricCritic, TypeSharedActor

def as_tensor(array, dtype=torch.float32):
    return torch.as_tensor(array, dtype=dtype)

actor = TypeSharedActor(hidden_dim=64)
critic = AgentCentricCritic(hidden_dim=64, num_heads=4)

with torch.no_grad():
    logits = actor(
        self_features=as_tensor(batch.self_features),
        ready_queue=as_tensor(batch.ready_queue),
        ready_mask=as_tensor(batch.ready_mask),
        other_cores=as_tensor(batch.other_cores),
        other_core_mask=as_tensor(batch.other_core_mask),
        system=as_tensor(batch.system),
        action_mask=as_tensor(batch.action_mask, dtype=torch.bool),
    )
    values = critic(
        self_features=as_tensor(batch.self_features),
        ready_queue=as_tensor(batch.ready_queue),
        ready_mask=as_tensor(batch.ready_mask),
        other_cores=as_tensor(batch.other_cores),
        other_core_mask=as_tensor(batch.other_core_mask),
        system=as_tensor(batch.system),
    )

print("logits:", logits.shape)
print("values:", values.shape)
print("sample logits row:", logits[0].tolist())

In [ ]:
from src.rl.trainer import ACACConfig, ACACTrainer, TorchACACPolicy

env = SchedulerEnv(
    core_config={CoreType.P: 1, CoreType.E: 1},
    workload_scenario=WorkloadScenario.BALANCED,
    arrival_rate=0.5,
    episode_time=30.0,
    max_tasks=4,
    seed=3,
)
rollout = collect_episode(env, FirstValidPolicy(), seed=3)
policy = TorchACACPolicy(ACACConfig(hidden_dim=64, critic_heads=4))
trainer = ACACTrainer(policy)
stats = trainer.update(rollout)
stats

In [ ]:
from torch.distributions import Categorical

class TorchActorPolicy:
    def __init__(self):
        self.actors = {core_type: TypeSharedActor(hidden_dim=64) for core_type in CoreType}

    def act(self, batch: AgentBatch):
        actions = {agent_id: 0 for agent_id in batch.agent_ids}
        log_probs = {agent_id: 0.0 for agent_id in batch.agent_ids}
        for row, agent_id in enumerate(batch.agent_ids):
            if not bool(batch.decision_mask[row]):
                continue
            core_type = list(CoreType)[int(batch.core_type_indices[row])]
            actor = self.actors[core_type]
            with torch.no_grad():
                row_logits = actor(
                    self_features=as_tensor(batch.self_features[row:row + 1]),
                    ready_queue=as_tensor(batch.ready_queue[row:row + 1]),
                    ready_mask=as_tensor(batch.ready_mask[row:row + 1]),
                    other_cores=as_tensor(batch.other_cores[row:row + 1]),
                    other_core_mask=as_tensor(batch.other_core_mask[row:row + 1]),
                    system=as_tensor(batch.system[row:row + 1]),
                    action_mask=as_tensor(batch.action_mask[row:row + 1], dtype=torch.bool),
                )[0]
                dist = Categorical(logits=row_logits)
                action = dist.sample()
            actions[agent_id] = int(action.item())
            log_probs[agent_id] = float(dist.log_prob(action).item())
        return actions, log_probs

env = SchedulerEnv(
    core_config={CoreType.P: 2, CoreType.E: 2},
    workload_scenario=WorkloadScenario.BALANCED,
    arrival_rate=0.4,
    episode_time=40.0,
    max_tasks=8,
    seed=7,
)
buffer = collect_episode(env, TorchActorPolicy(), seed=7)
print("sampled-policy transitions:", len(buffer))
print("completed:", len(env.completed_tasks), "/", len(env.tasks))